# Phase 2 Notebook: Candidate Generation Wikidata

## Principles
The goal is to find wikidata candidates for the mentions identified in the previous step. Example numbers:
* episodes: 2038
* publications: 2548
* seasons: 17
* persons: 10098
* topics: 10713

The goal is to be systematic and carefull, to put as little burden onto wikidata as possible. We should try to infer as much information from links as possible before we do any other searches, such as label searches or manual searches for the remaining items without candidates. 

## Approach (Simple Tree Expansion Workflow)
1. Load the broadcasting program seed(s) from `data/00_setup/broadcasting_programs.csv` and resolve their Wikidata Q-IDs.
2. For each seed Q-ID, check local cache first.
   - If cached entity data exists, use it.
   - If not cached, query Wikidata once and store the raw result as a new separate file.
3. Expand the Wikidata tree from each known Q-ID.
   - Expand outlinks (claims from the entity to other entities/properties).
   - Expand inlinks (entities that point to the current entity).
   - Persist each expansion query result as its own file before any aggregation.
4. Stage A is graph-first and authoritative: expand by graph predicates and seed/core-class rules (not by literal matches).
5. Stage B runs only for unresolved targets and performs scoped fallback string matching.
6. Eligible fallback discoveries are re-checked and re-entered into graph expansion.
7. Maintain an aggregated index for fast lookup, but always keep per-query raw files as the source of truth.
   - If an aggregate file is corrupted, rebuild it from individual query result files without data loss.

### Cache And Storage Principles
- Cache-first: never hit Wikidata if sufficient local query results already exist.
- Append-only raw query storage: one file per query result.
- Rebuildable aggregates: derived files can be regenerated from raw query files at any time.
- Idempotent reruns: rerunning the workflow should reuse existing raw files and only fetch missing queries.

## 1) Project Setup
Resolve repository paths and make the source package importable in this notebook session.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "data").exists() and (cur / "speakermining" / "src").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Repository root not found.")


ROOT = find_repo_root(Path.cwd())
SRC = ROOT / "speakermining" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

## 2) Configure Workflow Parameters
Set rate limiting, cache behavior, and expansion limits before running discovery.
Use `max_queries_per_run` to control whether the notebook stays cache-only or performs live discovery queries.

In [ ]:
# Workflow Configuration
config = {
    "max_depth": 2,                          # Max tree expansion depth (0=seeds only, 1=neighbors, 2=2nd neighbors, etc.)
    "max_nodes": 50000,                      # Max total nodes to expand
    "max_queries_per_run": 0,               # Query budget semantics: -1=unlimited, 0=no network queries, >0=finite cap
    "max_neighbors_per_match": 200,          # Cap neighbors enqueued per matched node (0=unlimited)
    "query_timeout_seconds": 5,              # Timeout per HTTP request
    "query_delay_seconds": 1.2,              # Delay between queries (rate limiting)
    "network_progress_every": 50,            # Emit runtime progress after every N network calls (0=off)
    "inlinks_limit": 200,                    # Max inlinks per entity (SPARQL LIMIT)
    "cache_max_age_days": 365,               # Cache age threshold (days); older records are refreshed
    "fallback_enabled_mention_types": {      # Toggle fallback matching per mention type (set True/False)
        "person": True,
        "organization": False,
        "episode": False,
        "season": False,
        "topic": False,
        "broadcasting_program": False,
    },
}

print("Workflow Configuration:")
for key, value in config.items():
    if key == "fallback_enabled_mention_types" and isinstance(value, dict):
        enabled = [name for name, flag in value.items() if bool(flag)]
        print(f"  {key}: {value}")
        print(f"  fallback_enabled (resolved): {enabled}")
    else:
        print(f"  {key}: {value}")


## 3) Decide Resume Mode
Define whether this run should append from latest checkpoint, restart from seed 1, or revert one seed and continue.

In [ ]:
from process.candidate_generation.wikidata.checkpoint import decide_resume_mode

resume_mode = "append"  # options: append, restart, revert
resume_decision = decide_resume_mode(ROOT, resume_mode)
print("Resume decision:")
print(resume_decision)

## 4) Bootstrap and Load Broadcasting Program Seeds
Initialize required artifacts from setup data and load seed instances for expansion.

In [ ]:
from process.candidate_generation.broadcasting_program import load_broadcasting_program_seeds
from process.candidate_generation.wikidata.bootstrap import initialize_bootstrap_files, load_core_classes, load_seed_instances
from process.candidate_generation.wikidata.common import canonical_qid

setup_core_classes = load_core_classes(ROOT)
setup_seeds, skipped_seed_rows = load_seed_instances(ROOT)
initialize_bootstrap_files(ROOT, setup_core_classes, setup_seeds)

seeds = setup_seeds or load_broadcasting_program_seeds(ROOT)
seed_qids = [canonical_qid(seed.get("wikidata_id", "")) for seed in seeds]
seed_qids = [qid for qid in seed_qids if qid]

print(f"Bootstrap initialized with {len(setup_core_classes)} core classes and {len(setup_seeds)} valid seed rows")
print(f"Skipped seed rows due to invalid wikidata_id: {len(skipped_seed_rows)}")
print(f"Loaded {len(seeds)} broadcasting programs as seeds")
for seed in seeds:
    qid = canonical_qid(seed.get("wikidata_id", ""))
    if qid:
        print(f"\t• {seed.get('label', seed.get('name', 'N/A'))} ({qid})")


## 5) Build Mention Targets From Phase 2 Lookup Outputs
Target construction is delegated to a process module for maintainability.
Sources:
- `data/20_candidate_generation/episodes.csv` for episode, season, person, topic, and organization-like publication program mentions.
- `data/20_candidate_generation/broadcasting_programs.csv` for explicit broadcasting program entities (show-level).

In [ ]:
from process.candidate_generation.wikidata.candidate_targets import build_targets_from_phase2_lookup

phase2_dir = ROOT / "data" / "20_candidate_generation"
required_inputs = [
    phase2_dir / "episodes.csv",
    phase2_dir / "broadcasting_programs.csv",
]
missing_inputs = [path for path in required_inputs if not path.exists()]

if missing_inputs:
    missing_list = "\n".join(f"- {path}" for path in missing_inputs)
    raise FileNotFoundError(
        "Notebook 21 prerequisite missing: required Phase 2 candidate-generation inputs were not found.\n"
        f"Missing files:\n{missing_list}\n\n"
        "Please run speakermining/src/process/notebooks/20_candidate_generation_wikibase.ipynb first "
        "to generate these files, then rerun this cell."
    )

all_target_rows, mention_stats, target_df = build_targets_from_phase2_lookup(ROOT)
total_mentions = len(all_target_rows)

print("Loaded matching targets from Phase 2 lookup outputs")
print(f"target rows for matching: {total_mentions}")
print("\nMention Targets by Type:")
print(mention_stats.to_string())

print("\nTargets:")
display(target_df)

## 6) Execute Graph-First Expansion Stage
Run deterministic seed-ordered graph expansion (authoritative stage).

In [ ]:
from time import perf_counter

from process.candidate_generation.wikidata.expansion_engine import ExpansionConfig, run_graph_expansion_stage
from process.candidate_generation.wikidata.bootstrap import load_core_classes

print("[notebook] Step 6 start: graph-first expansion")
print(f"Target rows from episodes.csv: {len(all_target_rows)}")

expansion_config = ExpansionConfig(
    max_depth=config["max_depth"],
    max_nodes=config["max_nodes"],
    total_query_budget=config["max_queries_per_run"],
    per_seed_query_budget=config["max_queries_per_run"],
    query_timeout_seconds=config["query_timeout_seconds"],
    query_delay_seconds=config["query_delay_seconds"],
    network_progress_every=int(config.get("network_progress_every", 50) or 0),
    inlinks_limit=config["inlinks_limit"],
    cache_max_age_days=config["cache_max_age_days"],
    max_neighbors_per_node=config["max_neighbors_per_match"],
)

step6_t0 = perf_counter()
print("[notebook] -> run_graph_expansion_stage")
result = run_graph_expansion_stage(
    ROOT,
    seeds=seeds,
    targets=all_target_rows,
    core_class_qids={canonical_qid(row.get('wikidata_id', '')) for row in load_core_classes(ROOT) if canonical_qid(row.get('wikidata_id', ''))},
    config=expansion_config,
    requested_mode=resume_decision["mode"],
)
step6_elapsed = perf_counter() - step6_t0
summary = result.checkpoint_stats

print("\n[notebook] Step 6 complete")
print(f"[notebook] Step 6 elapsed seconds: {step6_elapsed:.2f}")
print("Execution Summary:")
print("=" * 50)
for key, value in summary.items():
    if not key.endswith("_csv"):
        print(f"  {key:<30} {value}")

## 6.5) Run Node Integrity Pass
Verify and repair graph integrity before fallback:
- ensure known nodes have minimal discovery payload
- expand discovered nodes that are eligible but still unexpanded
- persist diagnostics on regenerated nodes for follow-up bug analysis

In [ ]:
import json
from datetime import datetime, timezone
import pandas as pd

from process.candidate_generation.wikidata.node_integrity import (
    NodeIntegrityConfig,
    run_node_integrity_pass,
 )

print("[notebook] Step 6.5 start: node integrity pass")
raw_max_queries_per_run = int(config.get("max_queries_per_run", 0))
if raw_max_queries_per_run < -1:
    raise ValueError("config['max_queries_per_run'] must be -1, 0, or a positive integer")

stage_a_queries_this_run = int(
    result.checkpoint_stats.get(
        "stage_a_network_queries_this_run",
        result.checkpoint_stats.get("stage_a_network_queries", result.checkpoint_stats.get("total_queries", 0)),
    ) or 0
)

if raw_max_queries_per_run == -1:
    node_integrity_discovery_budget = -1
    node_integrity_total_expansion_budget = -1
    node_integrity_per_node_budget = -1
    node_integrity_budget_label = "unlimited"
else:
    remaining_after_stage_a = max(0, raw_max_queries_per_run - stage_a_queries_this_run)
    node_integrity_discovery_budget = remaining_after_stage_a
    node_integrity_total_expansion_budget = remaining_after_stage_a
    node_integrity_per_node_budget = remaining_after_stage_a
    node_integrity_budget_label = str(remaining_after_stage_a)

print(
    f"[notebook] Step 6.5 budget planning: stage_a_queries_this_run={stage_a_queries_this_run}, "
    f"node_integrity_budget={node_integrity_budget_label}"
)
node_integrity_config = NodeIntegrityConfig(
    cache_max_age_days=config["cache_max_age_days"],
    query_timeout_seconds=config["query_timeout_seconds"],
    query_delay_seconds=config["query_delay_seconds"],
    network_progress_every=int(config.get("network_progress_every", 50) or 0),
    discovery_query_budget=node_integrity_discovery_budget,
    per_node_expansion_query_budget=node_integrity_per_node_budget,
    total_expansion_query_budget=node_integrity_total_expansion_budget,
    inlinks_limit=config["inlinks_limit"],
    max_nodes_to_expand=0,
)

node_integrity_t0 = perf_counter()
node_integrity_result = run_node_integrity_pass(
    ROOT,
    config=node_integrity_config,
    seed_qids={canonical_qid(seed.get("wikidata_id", "")) for seed in seeds if canonical_qid(seed.get("wikidata_id", ""))},
    core_class_qids={canonical_qid(row.get("wikidata_id", "")) for row in load_core_classes(ROOT) if canonical_qid(row.get("wikidata_id", ""))},
)
node_integrity_elapsed = perf_counter() - node_integrity_t0

run_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
diagnostics_dir = ROOT / "data" / "20_candidate_generation" / "wikidata" / "node_integrity"
docs_dir = ROOT / "documentation" / "context" / "node_integrity"
diagnostics_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)

summary_record = {
    "run_timestamp_utc": run_ts,
    "known_qids": node_integrity_result.known_qids,
    "checked_qids": node_integrity_result.checked_qids,
    "repaired_discovery_qids": node_integrity_result.repaired_discovery_qids,
    "newly_discovered_qids": len(node_integrity_result.newly_discovered_qids),
    "eligible_unexpanded_qids": len(node_integrity_result.eligible_unexpanded_qids),
    "expanded_qids": len(node_integrity_result.expanded_qids),
    "network_queries_discovery": node_integrity_result.network_queries_discovery,
    "network_queries_expansion": node_integrity_result.network_queries_expansion,
    "total_network_queries": node_integrity_result.total_network_queries,
    "elapsed_seconds": round(node_integrity_elapsed, 3),
}

event_rows = []
for qid in sorted(node_integrity_result.repaired_qids):
    event_rows.append({"qid": qid, "event": "repaired_discovery"})
for qid in sorted(node_integrity_result.newly_discovered_qids):
    event_rows.append({"qid": qid, "event": "newly_discovered"})
for qid in sorted(node_integrity_result.expanded_qids):
    event_rows.append({"qid": qid, "event": "expanded_by_integrity"})

events_df = pd.DataFrame(event_rows, columns=["qid", "event"])
if not events_df.empty:
    events_df = events_df.sort_values(["event", "qid"]).reset_index(drop=True)

summary_json_path = diagnostics_dir / f"node_integrity_summary_{run_ts}.json"
summary_csv_path = diagnostics_dir / f"node_integrity_summary_{run_ts}.csv"
events_csv_path = diagnostics_dir / f"node_integrity_events_{run_ts}.csv"
doc_md_path = docs_dir / f"node_integrity_{run_ts}.md"

summary_json_path.write_text(
    json.dumps({"summary": summary_record, "materialize_stats": node_integrity_result.materialize_stats}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
pd.DataFrame([summary_record]).to_csv(summary_csv_path, index=False, encoding="utf-8")
events_df.to_csv(events_csv_path, index=False, encoding="utf-8")

md_lines = [
    "# Node Integrity Report",
    "",
    f"timestamp_utc: {run_ts}",
    "",
    "## Summary",
    "",
    f"- known_qids: {summary_record['known_qids']}",
    f"- checked_qids: {summary_record['checked_qids']}",
    f"- repaired_discovery_qids: {summary_record['repaired_discovery_qids']}",
    f"- newly_discovered_qids: {summary_record['newly_discovered_qids']}",
    f"- eligible_unexpanded_qids: {summary_record['eligible_unexpanded_qids']}",
    f"- expanded_qids: {summary_record['expanded_qids']}",
    f"- network_queries_discovery: {summary_record['network_queries_discovery']}",
    f"- network_queries_expansion: {summary_record['network_queries_expansion']}",
    f"- total_network_queries: {summary_record['total_network_queries']}",
    f"- elapsed_seconds: {summary_record['elapsed_seconds']}",
    "",
    "## Artifact Paths",
    "",
    f"- summary_json: {summary_json_path.relative_to(ROOT)}",
    f"- summary_csv: {summary_csv_path.relative_to(ROOT)}",
    f"- events_csv: {events_csv_path.relative_to(ROOT)}",
    "",
    "## Regenerated / Expanded QIDs",
    "",
]
if events_df.empty:
    md_lines.append("- none")
else:
    for _, row in events_df.iterrows():
        md_lines.append(f"- {row['event']}: {row['qid']}")

doc_md_path.write_text("\n".join(md_lines) + "\n", encoding="utf-8")

print(f"[notebook] Step 6.5 complete in {node_integrity_elapsed:.2f}s")
print("Node integrity summary:")
for key in [
    "known_qids",
    "checked_qids",
    "repaired_discovery_qids",
    "newly_discovered_qids",
    "eligible_unexpanded_qids",
    "expanded_qids",
    "network_queries_discovery",
    "network_queries_expansion",
    "total_network_queries",
]:
    print(f"  {key}: {summary_record[key]}")

print("\nPersistent diagnostics written:")
print(f"  - {summary_json_path.relative_to(ROOT)}")
print(f"  - {summary_csv_path.relative_to(ROOT)}")
print(f"  - {events_csv_path.relative_to(ROOT)}")
print(f"  - {doc_md_path.relative_to(ROOT)}")

if events_df.empty:
    print("\nNo regenerated or expanded nodes were required in this pass.")
else:
    print("\nNodes touched by integrity pass:")
    display(events_df)

## 7) Build Unresolved Handoff
Prepare unresolved targets and class-scope hints for fallback stage.

In [ ]:
unresolved_targets = result.unresolved_targets
core_class_rows = load_core_classes(ROOT)
class_by_filename = {str(row.get('filename', '')): canonical_qid(row.get('wikidata_id', '')) for row in core_class_rows}

base_scope_map = {
    'person': class_by_filename.get('persons', ''),
    'organization': class_by_filename.get('organizations', ''),
    'episode': class_by_filename.get('episodes', ''),
    'season': class_by_filename.get('series', ''),
    'topic': class_by_filename.get('topics', ''),
    'broadcasting_program': class_by_filename.get('broadcasting_programs', ''),
}
raw_fallback_types = config.get('fallback_enabled_mention_types', {'person': True})
if isinstance(raw_fallback_types, dict):
    enabled_mention_types = {
        str(name).strip().lower()
        for name, enabled in raw_fallback_types.items()
        if str(name).strip() and bool(enabled)
    }
else:
    enabled_mention_types = {
        str(value).strip().lower()
        for value in raw_fallback_types
        if str(value).strip()
    }
if not enabled_mention_types:
    enabled_mention_types = {'person'}

class_scope_hints = {
    mention_type: [qid]
    for mention_type, qid in base_scope_map.items()
    if mention_type in enabled_mention_types and qid
}

print(f'Unresolved targets handed to fallback: {len(unresolved_targets)}')
print(f'Fallback enabled mention types: {sorted(enabled_mention_types)}')
print('Class-scope hints:')
for key in sorted(class_scope_hints):
    print(f'  {key}: {class_scope_hints[key]}')

## 8) Run Fallback String Matching Stage
Fallback runs only on unresolved targets and uses class-scope narrowing.

In [ ]:
from time import perf_counter

from process.candidate_generation.wikidata.fallback_matcher import (
    enqueue_eligible_fallback_qids,
    merge_stage_candidates,
    run_fallback_string_matching_stage,
)

seed_qids = {canonical_qid(seed.get('wikidata_id', '')) for seed in seeds if canonical_qid(seed.get('wikidata_id', ''))}
stage_a_queries = int(
    result.checkpoint_stats.get(
        'stage_a_network_queries_this_run',
        result.checkpoint_stats.get('stage_a_network_queries', result.checkpoint_stats.get('total_queries', 0)),
    ) or 0
)
raw_total_query_budget = int(config.get('max_queries_per_run', 0))
if raw_total_query_budget < -1:
    raise ValueError("config['max_queries_per_run'] must be -1, 0, or a positive integer")
if raw_total_query_budget == -1:
    fallback_query_budget = -1
else:
    fallback_query_budget = max(0, raw_total_query_budget - stage_a_queries)
raw_fallback_types = config.get('fallback_enabled_mention_types', {'person': True})
if isinstance(raw_fallback_types, dict):
    fallback_enabled_types = [
        str(name).strip().lower()
        for name, enabled in raw_fallback_types.items()
        if str(name).strip() and bool(enabled)
    ]
else:
    fallback_enabled_types = [str(value).strip().lower() for value in raw_fallback_types if str(value).strip()]
if not fallback_enabled_types:
    fallback_enabled_types = ['person']
fallback_config = {
    'network_budget_remaining': fallback_query_budget,
    'max_queries_per_run': fallback_query_budget,
    'query_delay_seconds': float(config.get('query_delay_seconds', 1.0) or 0.0),
    'query_timeout_seconds': int(config.get('query_timeout_seconds', 30) or 30),
    'cache_max_age_days': int(config.get('cache_max_age_days', 365) or 365),
    'network_progress_every': int(config.get('network_progress_every', 50) or 0),
    'fallback_search_limit': 10,
    'fallback_search_languages': ['de', 'en'],
    'fallback_enabled_mention_types': fallback_enabled_types,
}
print('[notebook] Step 8 start: fallback string matching')
print(f'Stage A queries used (this run): {stage_a_queries}')
fallback_budget_label = 'unlimited' if fallback_query_budget == -1 else str(fallback_query_budget)
print(f'Fallback query budget remaining: {fallback_budget_label}')
print(f"Fallback progress interval: {fallback_config['network_progress_every']} calls")

step8_t0 = perf_counter()
print('[notebook] -> run_fallback_string_matching_stage')
fallback_result = run_fallback_string_matching_stage(
    ROOT,
    unresolved_targets=unresolved_targets,
    seeds=seed_qids,
    core_class_qids={qid for values in class_scope_hints.values() for qid in values},
    class_scope_hints=class_scope_hints,
    config=fallback_config,
 )
step8_elapsed = perf_counter() - step8_t0
merged_candidates = merge_stage_candidates(result.discovered_candidates, fallback_result.fallback_candidates)
print(f'[notebook] Step 8 complete in {step8_elapsed:.2f}s')
print(f'Fallback candidates: {len(fallback_result.fallback_candidates)}')
print(f'Eligible for re-entry expansion: {len(fallback_result.eligible_for_expansion_qids)}')
print(f'Ineligible fallback qids: {len(fallback_result.ineligible_qids)}')
print(f'Merged stage candidates (graph authoritative): {len(merged_candidates)}')

## 9) Re-check Eligibility and Expand Eligible Fallback Discoveries
Eligible fallback QIDs are re-entered into graph expansion with the same policy checks.

In [ ]:
reentry_summary = enqueue_eligible_fallback_qids(
    ROOT,
    candidate_qids=fallback_result.eligible_for_expansion_qids,
    seeds={canonical_qid(seed.get('wikidata_id', '')) for seed in seeds if canonical_qid(seed.get('wikidata_id', ''))},
    core_class_qids={qid for values in class_scope_hints.values() for qid in values},
    expansion_config=expansion_config,
)
print('Fallback re-entry summary:')
print(reentry_summary)

## 10) Review Deterministic Graph Artifacts
Inspect instances/triples/query inventory outputs from checkpoint materialization.

In [ ]:
import pandas as pd
from pathlib import Path

# Load deterministic materialization artifacts from projections folder
projections_dir = ROOT / "data" / "20_candidate_generation" / "wikidata" / "projections"
instances_path = projections_dir / "instances.csv"
triples_path = projections_dir / "triples.csv"
inventory_path = projections_dir / "query_inventory.csv"

if instances_path.exists():
    instances_df = pd.read_csv(instances_path)
    print(f"Loaded {len(instances_df)} rows from {instances_path.name}\n")

    print("Instances Preview:")
    display(instances_df)

    if triples_path.exists():
        triples_df = pd.read_csv(triples_path)
        print(f"Loaded {len(triples_df)} rows from {triples_path.name}")
        display(triples_df)

    if inventory_path.exists():
        inventory_df = pd.read_csv(inventory_path)
        print(f"Loaded {len(inventory_df)} rows from {inventory_path.name}")
        display(inventory_df)
else:
    print(f"Materialized instances file not found at {instances_path}")
